In [ ]:
# Imports
import sys
sys.path.append("../..")

import os
import torch
import torchaudio
from datasets import load_dataset, Audio, concatenate_datasets
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
import evaluate
from tqdm import tqdm
from utils.audio import load_common_voice_dataset_from_tsv

In [ ]:
# Config
LANGS = ["en", "hi"]
MODEL_NAME = "ai4bharat/indicwav2vec2-hindi-en"
SAMPLE_RATE = 16000

In [ ]:
# Load WER metric
wer_metric = evaluate.load("wer")

In [ ]:
# Load processor & model
processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
model = Wav2Vec2ForCTC.from_pretrained(MODEL_NAME)
model.eval()

In [ ]:
# Load English Dataset
print("Loading English dataset...")
english_dataset = load_dataset("mozilla-foundation/common_voice_13_0", "en", split="test")
english_dataset = english_dataset.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))
english_dataset = english_dataset.select(range(100))

# Load Hindi Dataset
print("Loading Hindi dataset...")
hindi_dataset_path = "../../full_datasets/asr/common_voice_hi/cv-corpus-22.0-2025-06-20/hi"
tsv_path = os.path.join(hindi_dataset_path, "test.tsv")
clips_path = os.path.join(hindi_dataset_path, "clips")

hindi_dataset = load_common_voice_dataset_from_tsv(tsv_path, clips_path)
hindi_dataset = hindi_dataset.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))
hindi_dataset = hindi_dataset.select(range(100))

In [ ]:
# Merge both datasets
dataset = concatenate_datasets([english_dataset, hindi_dataset])
print(f"🧾 Combined dataset size: {len(dataset)}")

In [ ]:
# Preprocessing function
def prepare_inputs(batch):
    inputs = processor(batch["audio"]["array"], sampling_rate=SAMPLE_RATE, return_tensors="pt", padding=True)
    with torch.no_grad():
        logits = model(**inputs).logits
    pred_ids = torch.argmax(logits, dim=-1)
    batch["pred_str"] = processor.batch_decode(pred_ids)[0]
    batch["target_str"] = batch["sentence"].lower()
    return batch

In [ ]:
# Run inference
print("Running inference...")
results = dataset.map(prepare_inputs)

In [ ]:
# Compute WER
preds = results["pred_str"]
targets = results["target_str"]

wer_score = wer_metric.compute(predictions=preds, references=targets)
print(f"\nWER on mixed-language test set: {wer_score:.4f}")